# CardioAgent: five-fold model comparison

This notebook clones code from GitHub and reads the five patient-aware folds from Google Drive. Select **Runtime > Change runtime type > T4 GPU** before running the CNN cells.

In [ ]:
# Code is cloned from GitHub.  The data remains separate in Google Drive.
REPOSITORY_URL = 'https://github.com/adem147/CaridoAgentOps.git'
PROJECT_DIR = '/content/CardioAgentOps'
!git clone "$REPOSITORY_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR/services/audio-engine"

In [ ]:
!pip install -q librosa scikit-learn mlflow onnx onnxruntime skl2onnx

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy only ourdataset/five_folds to this Drive folder; do not put it in GitHub.
FOLDS_DIR = '/content/drive/MyDrive/CardioAgentData/five_folds'

# Decision tree: five fair train/test rotations using all 149 features.
!python train_decision_tree.py --dataset-dir "$FOLDS_DIR"

# SVM: same folds and 149 features, with training-fold feature scaling.
!python train_svm.py --dataset-dir "$FOLDS_DIR" --C 1.0

In [ ]:
# Custom CNN: same folds, MFCC standardization fitted only on training folds,
# weighted loss, mini-batches, and GPU acceleration when available.
!python train_custom_cnn.py --dataset-dir "$FOLDS_DIR" --epochs 30 --batch-size 64 --learning-rate 0.001